## 1.Import thư viện 

In [1]:
import json
import ijson
import faiss
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from pathlib import Path

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

d:\miniconda3\envs\searchengine\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Config 

Khai báo file chunks đầu vào, thư mục embedding/index đầu ra và model E5 sẽ sử dụng. 

In [2]:
CHUNKS_PATH = Path("../data/chunks/corpus_chunks_v2_e5.json")
OUTPUT_DIR = Path("../data/embeddings/e5_base")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "intfloat/multilingual-e5-base"

### Cấu hình batch và kích thước xử lý 

In [3]:
BATCH_SIZE = 16 
MAX_LENGTH = 512
SHARD_SIZE = 10_000
MAX_RECORDS = None  
TEXT_COL = "raw_text"

### Load tokenizer va model E5

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("device:", device)
print("dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
).to(device)

model.eval()

device: cuda
dtype: torch.float16


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

### Mean pooling token embeddings

Định nghĩa cách gom embedding của từng token thành một vector đại diện cho cả đoạn text. Attention_mask giúp loại bỏ padding token khi tính trung bình. 

In [5]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state.float() * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

### Hàm Encode danh sách văn bản 
Biến text thành ma trận embedding. Mỗi batch sẽ được tokenize, đưa qua model, mean pooling, normalize L2 để Faiss có thể index/search ổn định

In [6]:
@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            outputs = model(**inputs)

        embeddings = mean_pooling(
            outputs.last_hidden_state,
            inputs["attention_mask"],
        )

        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(
            embeddings.cpu().numpy().astype("float32")
        )

        del inputs, outputs, embeddings

        if device == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_embeddings)

### Đọc file chunks dạng JSON array

Tạo generator đọc từng item trong file JSON lớn bằng ijson. Tránh load toàn bộ corpus vào RAM cùng lúc 

In [7]:
def iter_chunks_json_array(path):
    with open(path, "rb") as f: 
        for item in ijson.items(f, "item"):
            yield item 

### Lưu embedding shard và metadata

Lưu vector của một shard vào `.npy` và metadata tương ứng vào `.parquet`. Hai file có cùng `shard_id` để sau này ghép lại đúng thứ tự 

In [8]:
def save_shard(shard_id, embeddings, metadata_rows): 
    emb_path = OUTPUT_DIR / f"embeddings_part_{shard_id:05d}.npy"
    meta_path = OUTPUT_DIR / f"metadata_part_{shard_id:05d}.parquet"
    embeddings = np.asarray(embeddings, dtype="float32")
    metadata=pd.DataFrame(metadata_rows)
    
    np.save(emb_path,embeddings) 
    metadata.to_parquet(meta_path, index=False) 
    
    print("saved:", emb_path, embeddings.shape)
    print("saved:", meta_path, metadata.shape)

### Tạo embedding cho corpus

Lặp qua các chunk, thêm prefix `passage: ` theo chuẩn E5, encode theo batch, gom thành shard và lưu ra disk. 

In [9]:
batch_texts = []
batch_meta = []

shard_embeddings = []
shard_metadata = []

shard_id = 0
total_records = 0
total_embedded = 0

for item in tqdm(iter_chunks_json_array(CHUNKS_PATH)):
    if MAX_RECORDS is not None and total_records >= MAX_RECORDS:
        break
    
    total_records += 1 
    
    text = str(item.get(TEXT_COL, "")).strip()
    
    if not text:
        continue 
    
    batch_texts.append("passage: " + text)
    batch_meta.append({
        "doc_id": item.get("doc_id"),
        "chunk_id": item.get("chunk_id"),
        "url": item.get("url"),
        "topic": item.get("topic"),
        "text": text,
    })
    
    if len(batch_texts) >= BATCH_SIZE : 
        embeddings = encode_texts(batch_texts)
        shard_embeddings.append(embeddings)
        shard_metadata.extend(batch_meta)
        
        total_embedded += len(batch_texts) 
        batch_texts = []
        batch_meta = []
        
        if len(shard_metadata) >= SHARD_SIZE:
            shard_embeddings = np.vstack(shard_embeddings)

            save_shard(
                shard_id=shard_id,
                embeddings=shard_embeddings,
                metadata_rows=shard_metadata,
            )

            shard_id += 1
            shard_embeddings = []
            shard_metadata = []

if batch_texts:
    embeddings = encode_texts(batch_texts)

    shard_embeddings.append(embeddings)
    shard_metadata.extend(batch_meta)

    total_embedded += len(batch_texts)
    
if shard_metadata:
    shard_embeddings = np.vstack(shard_embeddings)

    save_shard(
        shard_id=shard_id,
        embeddings=shard_embeddings,
        metadata_rows=shard_metadata,
    )

print("total_records:", total_records)
print("total_embedded:", total_embedded)

0it [00:00, ?it/s]

10000it [02:04, 27.81it/s]

saved: ..\data\embeddings\e5_base\embeddings_part_00000.npy (10000, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00000.parquet (10000, 5)


20016it [04:11, 68.11it/s] 

saved: ..\data\embeddings\e5_base\embeddings_part_00001.npy (10000, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00001.parquet (10000, 5)


...

saved: ..\data\embeddings\e5_base\embeddings_part_00053.npy (10000, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00053.parquet (10000, 5)


550000it [2:00:23, 47.14it/s] 

saved: ..\data\embeddings\e5_base\embeddings_part_00054.npy (10000, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00054.parquet (10000, 5)


553863it [2:01:09, 76.19it/s] 


saved: ..\data\embeddings\e5_base\embeddings_part_00055.npy (3863, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00055.parquet (3863, 5)
total_records: 553863
total_embedded: 553863


## 3. Build FAISS Index

### Kiểm tra các shard đã tạo 

Tìm tất cả các file embedding và metadata đã lưu. Sau đó in số shard để đảm bảo dữ liệu đầu vào cho bước build FAISS là đầy đủ 

In [10]:
embedding_files = sorted(OUTPUT_DIR.glob("embeddings_part_*.npy"))
metadata_files = sorted(OUTPUT_DIR.glob("metadata_part_*.parquet"))

print("embedding shards:", len(embedding_files))
print("metadata shards:", len(metadata_files))

embedding shards: 56
metadata shards: 56


### Build FAISS index từ embedding shards

Đọc từng shard embedding, tạo `IndexFlatIP`, add vector vào index và ghep metadata lại thành một bảng duy nhất. Vì một vector đã normalize nên inner product tương đương với cosine similarity 

In [11]:
index = None 
all_metadata = []
offset = 0 

for emb_file, meta_file in tqdm(list(zip(embedding_files, metadata_files))):
    embeddings = np.load(emb_file).astype("float32")
    metadata = pd.read_parquet(meta_file)
    
    if index is None: 
        dim = embeddings.shape[1] 
        index = faiss.IndexFlatIP(dim)
        
    metadata.insert(
        0,
        "embedding_index",
        np.arange(offset, offset+len(metadata))
    )
    index.add(embeddings)
    all_metadata.append(metadata)

    offset += len(metadata)

metadata = pd.concat(all_metadata, ignore_index=True)
print("faiss vectors:", index.ntotal)
print("metadata rows:", len(metadata))

100%|██████████| 56/56 [00:26<00:00,  2.09it/s]


faiss vectors: 553863
metadata rows: 553863


### Khai báo đường dẫn artifact semantic. 
Đặt tên file cho FAISS Index, metadata tổng hợp và config embedding. Các artifact này sẽ được load lại khi search mà không cần encode corpus lại 

In [12]:
INDEX_PATH = OUTPUT_DIR / "semantic_e5_base.faiss"
METADATA_PATH = OUTPUT_DIR / "chunks_metadata.parquet"
CONFIG_PATH = OUTPUT_DIR / "embedding_config.json"

### Ghi index và metadata ra disk. 

Lưu FAISS index vào `.faiss` và metadata vào `.parquet`. Đây là kết quả chính của bước indexing semantic

In [13]:
faiss.write_index(index, str(INDEX_PATH))
metadata.to_parquet(METADATA_PATH, index=False)

### Lưu config embedding

Ghi lại thông tin quan trọng của pipeline: model, prefix document/query, maxlength, metric FAISS, kích thước vector và số vector. Config này giúp search về sau dùng đúng cách encode ban đầu

In [14]:
config = {
    "model_name": MODEL_NAME,
    "text_column": TEXT_COL,
    "document_prefix": "passage: ",
    "query_prefix": "query: ",
    "max_length": MAX_LENGTH,
    "normalize_embeddings": True,
    "faiss_metric": "inner_product",
    "embedding_dim": int(index.d),
    "num_vectors": int(index.ntotal),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

config

{'model_name': 'intfloat/multilingual-e5-base',
 'text_column': 'raw_text',
 'document_prefix': 'passage: ',
 'query_prefix': 'query: ',
 'max_length': 512,
 'normalize_embeddings': True,
 'faiss_metric': 'inner_product',
 'embedding_dim': 768,
 'num_vectors': 553863}

## 4. Search Thử 

### Load Index để search thử 

Đọc lại FAISS Index và metadata đã lưu. Từ đay có thể search trực tiếp mà không cần build lại index trong RAM

In [15]:
index = faiss.read_index(str(INDEX_PATH))
metadata = pd.read_parquet(METADATA_PATH) 

### Hàm semantic search cơ bản 

Encode query với prefix `query: `, search top-k vector gần nhất trong FAISS, lấy metadata theo index trả về và gắn thêm score để hiển thị kết quả 

In [16]:
def semantic_search(query, top_k = 5) : 
    query_text = "query: "  + query 
    
    query_embedding = encode_texts(
        [query_text],
        batch_size=1,
        max_length=MAX_LENGTH,
    )
    scores, ids = index.search(query_embedding, top_k)

    results = []
    
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue

        row = metadata.iloc[idx].to_dict()
        row["score"] = float(score)
        results.append(row)

    return pd.DataFrame(results)

### Ví dụ search 

In [17]:
semantic_search("cướp tiệm vàng", top_k=5)

,embedding_index,doc_id,chunk_id,url,topic,text,score
0,232041,66349,66349_2,https://www.24h.com.vn/phi-thuong-ky-quac/chu-...,Phi thường - kỳ quặc,Chủ tiệm vàng phản ứng cực nhanh khống chế tên...,0.889356
1,4586,1290,1290_2,https://dantri.com.vn/xa-hoi/nghi-pham-cuop-ti...,Xã hội,Nghi phạm cướp tiệm vàng buông súng sau khi gặ...,0.887866
2,1636,445,445_2,https://nld.com.vn/thoi-su/bo-cong-an-yeu-cau-...,Trong nước,Bộ Công an: Miễn giảm toàn bộ viện phí cho sin...,0.883723
3,242826,69586,69586_1,https://docbao.vn/video/gay-can-man-khong-che-...,Video,Gay cấn màn khống chế đối tượng cầm dao định c...,0.883457
4,6509,1865,1865_1,https://nld.com.vn/thoi-su/doi-tuong-dung-sung...,Trong nước,Đối tượng dùng súng AK tấn công cướp tiệm vàng...,0.882966


In [18]:
semantic_search("quân đội Trung Quốc ở Thái Bình Dương", top_k=5)

,embedding_index,doc_id,chunk_id,url,topic,text,score
0,91017,25764,25764_0,https://dantri.com.vn/the-gioi/my-canh-bao-ve-...,Thế giới,Mỹ cảnh báo về quân đội Trung Quốc | Báo Dân t...,0.872774
1,498761,144393,144393_1,https://www.24h.com.vn/tin-tuc-quoc-te/chuyen-...,Thế giới,Chuyên gia phân tích lợi thế quân sự của Mỹ và...,0.868660
2,167125,47754,47754_1,https://thanhnien.vn/khi-trung-quoc-tham-vong-...,Thế giới,Khi Trung Quốc tham vọng thay đổi trật tự thế ...,0.866906
3,92850,26316,26316_0,https://thanhnien.vn/tuong-my-ra-canh-bao-ve-q...,Thế giới,Tướng Mỹ ra cảnh báo về quân đội Trung Quốc. T...,0.865307
4,87576,24870,24870_2,https://tuoitre.vn/tuong-my-trung-quoc-ngay-ca...,Thế giới,Tướng Mỹ: 'Trung Quốc ngày càng hung hăng ở Th...,0.861874


## Semantic Search helpers

### Hàm load artifact semantic đã lưu 

Đóng gói việc load FAISS index, metadata và config. Hữu ích khi mở lại noteook và chỉ muốn searhc, không muốn chạy lại bước tạo embedding. 

In [19]:
def load_semantic_artifacts(output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    index_path = output_dir / "semantic_e5_base.faiss"
    metadata_path = output_dir / "chunks_metadata.parquet"
    config_path = output_dir / "embedding_config.json"

    loaded_index = faiss.read_index(str(index_path))
    loaded_metadata = pd.read_parquet(metadata_path)

    with open(config_path, "r", encoding="utf-8") as f:
        loaded_config = json.load(f)

    print("index vectors:", loaded_index.ntotal)
    print("metadata rows:", len(loaded_metadata))
    print("model:", loaded_config["model_name"])

    return loaded_index, loaded_metadata, loaded_config

### Nạp artifact vào biến sử dụng 
Gắn kết quả vào `index`, `metadata`, `semantic_config` để các hàm search phía sau dùng chung 

In [20]:
index, metadata, semantic_config = load_semantic_artifacts()

index vectors: 553863
metadata rows: 553863
model: intfloat/multilingual-e5-base


### Hàm semantic search nâng cấp 

Mở rộng hàm search cơ bản: đọc prefix/max length từ config, lấy nhiều ứng viên hơn, hỗ trợ lọc theo topic, lọc theo min_score, và trả về DataFrame đã sắp  xếp theo score 

In [21]:
def semantic_search_v2(
    query,
    top_k=10,
    candidate_k=None,
    topic=None,
    min_score=None,
):
    if not query or not str(query).strip():
        return pd.DataFrame()

    candidate_k = candidate_k or max(top_k * 5, 50)
    candidate_k = min(candidate_k, index.ntotal)

    query_prefix = semantic_config.get("query_prefix", "query: ")
    query_text = query_prefix + str(query).strip()

    query_embedding = encode_texts(
        [query_text],
        batch_size=1,
        max_length=semantic_config.get("max_length", MAX_LENGTH),
    )

    scores, ids = index.search(query_embedding, candidate_k)
    rows = []

    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        if min_score is not None and score < min_score:
            continue

        row = metadata.iloc[int(idx)].to_dict()
        if topic is not None and row.get("topic") != topic:
            continue

        row["score"] = float(score)
        rows.append(row)

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

### Định dạng kết quả hiển thị 

Cắt ngắn cột `text` và chỉ giữu các cột quan trọng như score, doc_id, chunk_id, topic, text, url. Mục đíhc là làm bảng kết quả để đọc.

In [22]:
def show_search_results(results, max_text_chars=220):
    if results.empty:
        return results

    display_df = results.copy()
    display_df["text"] = display_df["text"].str.slice(0, max_text_chars)

    columns = [
        "score",
        "doc_id",
        "chunk_id",
        "topic",
        "text",
        "url",
    ]
    columns = [col for col in columns if col in display_df.columns]
    return display_df[columns]

### Chạy thử search nâng cấp 

In [29]:
results = semantic_search_v2("CÔng an bắt tên cướp tiệm vàng", top_k=5)
show_search_results(results)

,score,doc_id,chunk_id,topic,text,url
0,0.898551,1290,1290_2,Xã hội,Nghi phạm cướp tiệm vàng buông súng sau khi gặ...,https://dantri.com.vn/xa-hoi/nghi-pham-cuop-ti...
1,0.894848,14748,14748_0,Pháp luật,Truy bắt tên cướp tiệm vàng sau 12 giờ gây án....,https://zingnews.vn/truy-bat-ten-cuop-tiem-van...
2,0.891600,1129,1129_0,Thời sự,Đối tượng mang súng AK cướp tiệm vàng muốn tự ...,https://vietnamnet.vn/doi-tuong-mang-sung-ak-c...
3,0.888566,1155,1155_1,Pháp luật,"Nghi phạm cướp tiệm vàng mặc sắc phục công an,...",https://tuoitre.vn/nghi-pham-cuop-tiem-vang-ma...
4,0.887890,445,445_2,Trong nước,Bộ Công an: Miễn giảm toàn bộ viện phí cho sin...,https://nld.com.vn/thoi-su/bo-cong-an-yeu-cau-...


## Gộp kết quả theo bài viết 

Xử lý trường hợp một bài viết có nhiều chunk cùng lọt top. Hàm giữu chunk có score cao nhất cho mỗi `doc_id`, giúp màn hình kết quả đa dạng hơn theo bài viết.

In [24]:
def group_results_by_doc(results, top_k=5):
    if results.empty:
        return results

    grouped = (
        results.sort_values("score", ascending=False)
        .groupby("doc_id", as_index=False)
        .agg(
            score=("score", "max"),
            chunk_id=("chunk_id", "first"),
            topic=("topic", "first"),
            text=("text", "first"),
            url=("url", "first"),
        )
        .sort_values("score", ascending=False)
        .head(top_k)
    )

    return grouped.reset_index(drop=True)

### Ví dụ search và gộp theo bài viết 

In [30]:
chunk_results = semantic_search_v2("Nga và Ukraine", top_k=20)
doc_results = group_results_by_doc(chunk_results, top_k=5)
show_search_results(doc_results)

,score,doc_id,chunk_id,topic,text,url
0,0.862759,28054,28054_2,,Đằng sau quyết định bãi nhiệm hàng loạt quan c...,https://www.24h.com.vn/tin-tuc-quoc-te/dang-sa...
1,0.862628,12470,12470_6,Thế giới,Những điểm then chốt trong xung đột Nga - Ukra...,https://docbao.vn/the-gioi/nhung-diem-then-cho...
2,0.862621,124901,124901_10,Thế giới,Nhìn lại 4 tháng chiến dịch quân sự đặc biệt c...,https://vtv.vn/the-gioi/nhin-lai-4-thang-chien...
3,0.861903,9214,9214_0,Tư liệu,Các bên muốn gì trong xung đột Nga - Ukraine?....,https://vtc.vn/cac-ben-muon-gi-trong-xung-dot-...
4,0.861230,14928,14928_0,,Nga và Ukraine đang sử dụng những loại máy bay...,https://www.24h.com.vn/tin-tuc-quoc-te/nga-va-...
